# 05. Pandas 고급 분석 - 실습 문제

## 실습 안내
- 총 10개 문제
- 시계열 분석, 이동평균, OEE, SPC 등 실무 분석
- 종합적인 제조 데이터 분석 능력 배양
- 실제 공장 모니터링 대시보드 구축

## 데이터 로드 및 전처리

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# 데이터 불러오기
production_df = pd.read_csv('../data/05_production.csv', encoding='utf-8-sig')
quality_df = pd.read_csv('../data/07_quality_inspection.csv', encoding='utf-8-sig', na_values=['\\N'])
sensor_df = pd.read_csv('../data/08_sensor_data.csv', encoding='utf-8-sig')
operation_df = pd.read_csv('../data/06_equipment_operation.csv', encoding='utf-8-sig')
equipment_df = pd.read_csv('../data/01_equipment.csv', encoding='utf-8-sig')
maintenance_df = pd.read_csv('../data/10_maintenance_history.csv', encoding='utf-8-sig')

# 날짜/시간 변환
production_df['production_date'] = pd.to_datetime(production_df['production_date'])
production_df['start_time'] = pd.to_datetime(production_df['start_time'])
production_df['end_time'] = pd.to_datetime(production_df['end_time'])
quality_df['inspection_time'] = pd.to_datetime(quality_df['inspection_time'])
sensor_df['measurement_time'] = pd.to_datetime(sensor_df['measurement_time'])
operation_df['start_time'] = pd.to_datetime(operation_df['start_time'])
operation_df['end_time'] = pd.to_datetime(operation_df['end_time'])

print("데이터 로드 완료!")

데이터 로드 완료!


---
## 문제 1: 일별 생산량 추이 분석

**시나리오**: 일별 생산량 추이를 분석하여 생산 패턴을 파악하세요.

**요구사항**:
1. 일별로 다음 집계:
   - 생산 건수
   - 총 생산량
   - 총 불량수
   - 불량률 (%)
2. 처음 30일 데이터 출력
3. 불량률이 가장 높았던 날 찾기

**힌트**: `groupby('production_date')`, 계산 컬럼, `idxmax()`

In [2]:
# 여기에 코드 작성
daily_prod=production_df.groupby('production_date').agg({'production_id':'count',
                                              'actual_quantity':'sum',
                                               'defect_quantity':'sum',
                                               })

In [3]:
daily_prod.columns=['생산건수','총생산량','총불량수']

In [4]:
daily_prod['불량률']=(daily_prod['총불량수']/daily_prod['총생산량']*100).round(2)

In [5]:
daily_prod[daily_prod['불량률']==daily_prod['불량률'].max()]

,생산건수,총생산량,총불량수,불량률
production_date,,,,
2024-03-27,22,2496,386,15.46


---
## 문제 2: 센서 데이터 시간별 리샘플링

**시나리오**: INJ-001 설비의 센서 데이터를 시간 단위로 집계하세요.

**요구사항**:
1. sensor_df에서 equipment_id='INJ-001'인 데이터 필터링
2. measurement_time을 인덱스로 설정
3. 4시간 단위로 리샘플링하여 다음 집계:
   - temperature: 평균
   - pressure: 평균
   - vibration: 최대값
4. 처음 20개 결과 출력

**힌트**: `set_index()`, `resample('4H')`, `agg()`

In [6]:
sensor_df.head(2)

,sensor_id,equipment_id,measurement_time,temperature,pressure,vibration,current,voltage,rpm,created_at
0,1,INJ-001,2024-01-01,183.93,148.65,2.6838,48.05,218.83,1795.32,2026-01-30 00:45:52
1,2,INJ-002,2024-01-01,173.06,144.23,2.2576,39.17,211.38,1738.75,2026-01-30 00:45:52


In [7]:
sensor_df['equipment_id'].unique()

array(['INJ-001', 'INJ-002', 'PRESS-001', 'PRESS-002', 'ASM-001'],
      dtype=object)

In [8]:
sensor_df['measurement_time'].unique()

<DatetimeArray>
['2024-01-01 00:00:00', '2024-01-01 01:00:00', '2024-01-01 02:00:00',
 '2024-01-01 03:00:00', '2024-01-01 04:00:00', '2024-01-01 05:00:00',
 '2024-01-01 06:00:00', '2024-01-01 07:00:00', '2024-01-01 08:00:00',
 '2024-01-01 09:00:00',
 ...
 '2024-03-31 14:00:00', '2024-03-31 15:00:00', '2024-03-31 16:00:00',
 '2024-03-31 17:00:00', '2024-03-31 18:00:00', '2024-03-31 19:00:00',
 '2024-03-31 20:00:00', '2024-03-31 21:00:00', '2024-03-31 22:00:00',
 '2024-03-31 23:00:00']
Length: 2184, dtype: datetime64[ns]

In [9]:
sensor_df_INJ_001=sensor_df[sensor_df['equipment_id']=='INJ-001'].copy()

In [10]:
sensor_df_INJ_001=sensor_df_INJ_001.set_index('measurement_time') # 인덱스 변경하는 법!!

In [11]:
sensor_df_INJ_001

,sensor_id,equipment_id,temperature,pressure,vibration,current,voltage,rpm,created_at
measurement_time,,,,,,,,,
2024-01-01 00:00:00,1,INJ-001,183.93,148.65,2.6838,48.05,218.83,1795.32,2026-01-30 00:45:52
2024-01-01 01:00:00,6,INJ-001,179.26,155.23,2.6619,41.47,221.62,1792.30,2026-01-30 00:45:52
2024-01-01 02:00:00,11,INJ-001,179.78,146.19,2.3795,42.07,221.48,1805.22,2026-01-30 00:45:52
2024-01-01 03:00:00,16,INJ-001,181.30,143.83,2.4134,47.61,211.96,1803.69,2026-01-30 00:45:52
2024-01-01 04:00:00,21,INJ-001,179.96,152.57,2.4701,44.85,215.77,1769.70,2026-01-30 00:45:52
...,...,...,...,...,...,...,...,...,...
2024-03-31 19:00:00,10896,INJ-001,194.25,152.07,2.5405,45.38,215.45,1817.42,2026-01-30 00:45:52
2024-03-31 20:00:00,10901,INJ-001,192.63,145.85,2.4024,46.84,222.41,1773.20,2026-01-30 00:45:52
2024-03-31 21:00:00,10906,INJ-001,196.04,149.04,2.5558,41.28,227.03,1830.42,2026-01-30 00:45:52


In [12]:
sensor_df_INJ_001_4h=sensor_df_INJ_001.resample('4h').agg({'temperature':'mean',
                                       'pressure':'mean',
                                        'vibration':'max'})#resampling 하는법

In [13]:
sensor_df_INJ_001_4h=(sensor_df_INJ_001_4h).round(2)

In [14]:
sensor_df_INJ_001_4h

,temperature,pressure,vibration
measurement_time,,,
2024-01-01 00:00:00,181.07,148.48,2.68
2024-01-01 04:00:00,179.98,152.66,2.59
2024-01-01 08:00:00,182.58,146.17,2.54
2024-01-01 12:00:00,179.02,148.13,2.80
2024-01-01 16:00:00,180.33,151.48,2.64
...,...,...,...
2024-03-31 04:00:00,195.04,150.07,2.66
2024-03-31 08:00:00,196.25,148.43,2.69
2024-03-31 12:00:00,196.20,149.14,2.77


In [15]:
sensor_df_INJ_001_4h.describe() # 숫자 데이터의 통계를 알려줌

,temperature,pressure,vibration
count,546.000000,546.000000,546.000000
mean,187.510348,150.021886,2.712821
std,4.484908,1.669429,0.149778
min,178.240000,144.860000,2.320000
25%,183.772500,148.900000,2.610000
50%,187.470000,150.040000,2.700000
75%,191.347500,151.157500,2.810000
max,197.250000,155.270000,3.250000


In [16]:
sensor_df_INJ_001_4h[sensor_df_INJ_001_4h['temperature']==197.250000]

,temperature,pressure,vibration
measurement_time,,,
2024-03-31,197.25,153.55,2.98


---
## 문제 3: 생산량 이동평균 계산

**시나리오**: 일별 생산량에 이동평균을 적용하여 추세를 파악하세요.

**요구사항**:
1. 일별 총 생산량 집계
2. 7일 이동평균 계산
3. 30일 이동평균 계산
4. 원본, 7일MA, 30일MA를 함께 출력 (처음 60일)
5. 이동평균이 실제 생산량보다 높은 날 (하락 추세) 찾기

**힌트**: `rolling(window=n).mean()`, 비교 연산

In [17]:
# 여기에 코드 작성
daily_qty=production_df.groupby('production_date')['actual_quantity'].sum()

In [18]:
daily_qty_7=daily_qty.rolling(window=7).mean().round(2)## roiling함수

In [19]:
daily_qty_30=daily_qty.rolling(window=30).mean().round(2)

In [20]:
daily_qty_df=pd.DataFrame(data={'생산량':daily_qty,'7일 MA':daily_qty_7,'30일 MA': daily_qty_30 }) # 데이터 프레임 만드는 법

In [21]:
# 하락 추세 : 이동평균 > 실제 생산량 <-- 새로운 데이터 프레임에서 같은 행에 데이터가 들어야 함

In [22]:
daily_qty_df[daily_qty_df['7일 MA']>daily_qty_df['생산량']].shape

(48, 3)

---
## 문제 4: 전일 대비 생산량 변화 분석

**시나리오**: 일별 생산량의 전일 대비 변화를 분석하세요.

**요구사항**:
1. 일별 총 생산량 집계
2. 전일 생산량 추가 (`shift()`)
3. 전일 대비 증감량 계산 (`diff()`)
4. 전일 대비 변화율(%) 계산 (`pct_change()`)
5. 변화율이 가장 큰 날(증가) 상위 5개 출력
6. 변화율이 가장 작은 날(감소) 상위 5개 출력

**힌트**: shift, diff, pct_change, sort_values

In [23]:
# 여기에 코드 작성
compa_before=production_df.groupby('production_date')['actual_quantity'].sum()

In [24]:
compa_before_1=compa_before.shift()

In [25]:
compa_before_df=pd.DataFrame(data={'당일생산량':compa_before,'전일생산량':compa_before_1})

In [26]:
compa_before_diff=compa_before.diff()

In [27]:
compa_before_df['전일 대비 증감량']=compa_before_diff

In [28]:
compa_before_pct=compa_before.pct_change()*100

In [29]:
compa_before_pct

production_date
2024-01-01          NaN
2024-01-02    17.880139
2024-01-03   -22.352941
2024-01-04    27.597403
2024-01-05    -1.187447
                ...    
2024-03-27    17.073171
2024-03-28   -15.584936
2024-03-29    21.072615
2024-03-30   -19.639357
2024-03-31     4.048780
Name: actual_quantity, Length: 91, dtype: float64

In [30]:
compa_before_df['전일 대비 변화율']=compa_before_pct

In [31]:
compa_before_df.sort_values('전일 대비 변화율').head(5)

,당일생산량,전일생산량,전일 대비 증감량,전일 대비 변화율
production_date,,,,
2024-01-20,1722,2515.0,-793.0,-31.530815
2024-01-17,1813,2513.0,-700.0,-27.855153
2024-02-04,1861,2443.0,-582.0,-23.823168
2024-01-03,1848,2380.0,-532.0,-22.352941
2024-03-22,1946,2440.0,-494.0,-20.245902


In [32]:
compa_before_df.sort_values('전일 대비 변화율', ascending=False).head(5)

,당일생산량,전일생산량,전일 대비 증감량,전일 대비 변화율
production_date,,,,
2024-02-05,2684,1861.0,823.0,44.223536
2024-01-18,2496,1813.0,683.0,37.672366
2024-01-21,2282,1722.0,560.0,32.520325
2024-03-02,2781,2111.0,670.0,31.738513
2024-03-09,2433,1876.0,557.0,29.690832


---
## 문제 5: 센서 데이터 이상치 탐지 (3-Sigma)

**시나리오**: PRESS-001 설비의 압력 센서 이상치를 탐지하세요.

**요구사항**:
1. sensor_df에서 equipment_id='PRESS-001' 필터링
2. pressure 컬럼의 평균과 표준편차 계산
3. 3-Sigma 방식으로 상한/하한 계산:
   - 하한 = 평균 - 3×표준편차
   - 상한 = 평균 + 3×표준편차
4. 이상치 플래그 컬럼 생성
5. 이상치 건수 및 비율 출력
6. 이상치 데이터 처음 10개 출력

**힌트**: mean(), std(), 조건 연산

In [33]:
# 여기에 코드 작성
sensor_sigma=sensor_df[sensor_df['equipment_id']=='PRESS-001'].copy()

In [34]:
sensor_sigma['pressure'].mean()

np.float64(199.9600457875458)

In [35]:
sensor_sigma['pressure'].std()

3.3883257307462116

In [36]:
over_line=sensor_sigma['pressure'].mean()+3*sensor_sigma['pressure'].std()

In [37]:
under_line=sensor_sigma['pressure'].mean()-3*sensor_sigma['pressure'].std()

In [38]:
sensor_press_outlier =sensor_sigma[(sensor_sigma['pressure']<=under_line) | (sensor_sigma['pressure']>over_line)]

In [39]:
import numpy as np

sensor_press['anomaly_flag'] = np.where(
    (sensor_press['pressure'] >= lower) & (sensor_press['pressure'] <= upper),
    '통과',
    '이상'
)

In [40]:
sensor_sigma['플래그'] = np.where(((sensor_sigma['pressure']<=under_line) | (sensor_sigma['pressure']>over_line)),'이상','통과')

In [41]:
sensor_sigma[sensor_sigma['플래그']=='이상']

,sensor_id,equipment_id,measurement_time,temperature,pressure,vibration,current,voltage,rpm,created_at,플래그
827,828,PRESS-001,2024-01-07 21:00:00,83.32,212.63,3.1379,117.29,383.67,458.46,2026-01-30 00:45:52,이상
1467,1468,PRESS-001,2024-01-13 05:00:00,83.58,210.36,3.5575,119.84,377.42,490.40,2026-01-30 00:45:52,이상
4427,4428,PRESS-001,2024-02-06 21:00:00,81.04,212.10,3.2676,123.47,383.09,503.62,2026-01-30 00:45:52,이상
5402,5403,PRESS-001,2024-02-15 00:00:00,81.97,211.25,3.6710,119.35,381.96,486.34,2026-01-30 00:45:52,이상
5997,5998,PRESS-001,2024-02-19 23:00:00,83.74,188.69,3.8420,118.22,377.65,521.25,2026-01-30 00:45:52,이상
6862,6863,PRESS-001,2024-02-27 04:00:00,85.74,210.45,3.5355,124.14,388.72,495.46,2026-01-30 00:45:52,이상


sensor_press['anomaly_flag'] = '통과'

sensor_press.loc[
    (sensor_press['pressure'] < lower) | (sensor_press['pressure'] > upper),
    'anomaly_flag'
] = '이상'

In [42]:
sensor_sigma['플래그'] = '통과'

In [43]:
sensor_sigma.loc[((sensor_sigma['pressure']<=under_line) | (sensor_sigma['pressure']>over_line)),'플래그']='이상'

In [44]:
sensor_sigma[sensor_sigma['플래그']=='이상']

,sensor_id,equipment_id,measurement_time,temperature,pressure,vibration,current,voltage,rpm,created_at,플래그
827,828,PRESS-001,2024-01-07 21:00:00,83.32,212.63,3.1379,117.29,383.67,458.46,2026-01-30 00:45:52,이상
1467,1468,PRESS-001,2024-01-13 05:00:00,83.58,210.36,3.5575,119.84,377.42,490.40,2026-01-30 00:45:52,이상
4427,4428,PRESS-001,2024-02-06 21:00:00,81.04,212.10,3.2676,123.47,383.09,503.62,2026-01-30 00:45:52,이상
5402,5403,PRESS-001,2024-02-15 00:00:00,81.97,211.25,3.6710,119.35,381.96,486.34,2026-01-30 00:45:52,이상
5997,5998,PRESS-001,2024-02-19 23:00:00,83.74,188.69,3.8420,118.22,377.65,521.25,2026-01-30 00:45:52,이상
6862,6863,PRESS-001,2024-02-27 04:00:00,85.74,210.45,3.5355,124.14,388.72,495.46,2026-01-30 00:45:52,이상


---
## 문제 6: OEE (Overall Equipment Effectiveness) 계산

**시나리오**: 생산 데이터에서 설비별 OEE를 계산하세요.

**요구사항**:
1. production_df에서 다음 계산:
   - 양품률 = good_quantity / actual_quantity
   - 실제 작업시간(분) = (end_time - start_time)
   - 성능률 = (actual_quantity × cycle_time) / 실제작업시간
   - 가동률 = 1.0 (간소화)
   - OEE = 가동률 × 성능률 × 양품률 × 100
2. 설비별 평균 OEE 계산
3. OEE 순위 출력 (높은 순)
4. OEE가 70% 미만인 설비 찾기

**힌트**: 시간 차이 계산, 복합 계산, groupby

In [45]:
# 여기에 코드 작성
production_df['양품률']=production_df['good_quantity']/production_df['actual_quantity']*100

In [46]:
production_df['실제작업시간(분)']=(production_df['end_time']-production_df['start_time']).dt.total_seconds()/60

In [47]:
#성능률
production_df['성능률']=production_df['actual_quantity']*(production_df['cycle_time']/60)/ production_df['실제작업시간(분)']

In [48]:
production_df['성능률'].clip(upper=1.0)## 1보다 큰 값은 자르기

0       1.000000
1       1.000000
2       1.000000
3       1.000000
4       1.000000
          ...   
1867    0.999975
1868    1.000000
1869    1.000000
1870    1.000000
1871    1.000000
Name: 성능률, Length: 1872, dtype: float64

In [49]:
production_df['성능률'].clip(upper=1.0).min()

0.9999263125575684

In [50]:
production_df['OEE'] = 1.0 *production_df['성능률']*production_df['양품률']

In [51]:
production_df.groupby('equipment_id')['OEE'].mean().sort_values(ascending=False)

equipment_id
ASM-001      97.848563
INJ-002      97.536522
INJ-001      94.261464
PRESS-001    93.484886
PRESS-002    92.836838
Name: OEE, dtype: float64

---
## 문제 7: SPC 관리도 데이터 생성

**시나리오**: DASH-C 제품의 측정값에 대한 SPC X-bar 관리도를 생성하세요.

**요구사항**:
1. quality_df에서 product_code='DASH-C' 필터링
2. 일별로 measurement_value의 평균, 최소, 최대, 표준편차 계산
3. 전체 일별 평균의 중심선(CL) 계산
4. 관리상한선(UCL) = CL + 3×표준편차
5. 관리하한선(LCL) = CL - 3×표준편차
6. 관리 이탈 일자 찾기 (평균이 UCL 초과 또는 LCL 미만)
7. 처음 20일 데이터 출력

**힌트**: 날짜 추출, groupby, 통계 계산, 조건 필터링

In [59]:
# 여기에 코드 작성
quality_dash=quality_df[quality_df['product_code']=='DASH-C'].copy()

In [61]:
quality_dash['inspection_date']=quality_dash['inspection_time'].dt.date

In [64]:
daily_quality=(quality_dash.groupby('inspection_date')['measurement_value'].agg(['mean','min','max','std'])).round(2)

In [62]:
quality_dash.columns

Index(['inspection_id', 'production_id', 'equipment_id', 'product_code',
       'inspection_time', 'inspection_type', 'result', 'defect_code',
       'measurement_value', 'measurement_unit', 'inspector_id', 'lot_no',
       'sample_size', 'notes', 'created_at', 'inspection_date'],
      dtype='object')

In [70]:
center_line = daily_quality['mean'].mean() # 전체 평균  = 센터라인

In [71]:
std = daily_quality['mean'].std() # 전체 표준편차 

In [72]:
# 관리상한/ 하한
ucl = center_line + 3*std
lcl = center_line - 3*std

In [75]:
daily_quality['관리상태']= 'OK'
# 관리상태 컬럼을 만들되, 초기값은 OK로 셋팅하고, UCL과 LCL 밖의 데이터는 따로 표시

In [79]:
daily_quality.loc[daily_quality['mean']>ucl,'관리상태']='UCL초과'

In [ ]:
daily_quality.loc[daily_quality['mean']<lcl,'관리상태']='LCL미만'

In [84]:
daily_quality[daily_quality['관리상태']!='OK']

,mean,min,max,std,관리상태
inspection_date,,,,,


---
## 문제 8: 월별 생산 트렌드 및 변화율 분석

**시나리오**: 월별 생산 추이와 전월 대비 변화율을 분석하세요.

**요구사항**:
1. production_date에서 년-월 추출
2. 월별로 다음 집계:
   - 생산 건수
   - 총 생산량
   - 평균 불량률
   - 설비 가동 수 (equipment_id unique count)
3. 전월 대비 생산량 변화율(%) 계산
4. 전월 대비 불량률 변화(차이) 계산
5. 처음 12개월 데이터 출력

**힌트**: `dt.to_period('M')`, pct_change, diff

In [86]:
# 여기에 코드 작성
production_df['year_month']=production_df['production_date'].dt.to_period('M')

In [88]:
monthly_suaus=production_df.groupby('year_month').agg({'production_id':'count',
                                          'actual_quantity':'sum',
                                           'defect_quantity':'sum',
                                           'equipment_id':'nunique'})

In [89]:
monthly_suaus.columns=['생산건수','총생산량','총불량수','가동설비수']

In [95]:
monthly_suaus['불량률']=monthly_suaus['총불량수']/monthly_suaus['총생산량']*100

In [97]:
monthly_suaus['불량률변화']=monthly_suaus['불량률'].diff()

In [101]:
monthly_suaus['생산량변화율']=monthly_suaus['총생산량'].pct_change()*100

In [102]:
monthly_suaus

,생산건수,총생산량,총불량수,가동설비수,불량률,불량률변화,생산량변화율
year_month,,,,,,,
2024-01,626,69849,3667,5,5.249896,NaN,NaN
2024-02,602,66390,6811,5,10.259075,5.009179,-4.952111
2024-03,644,70365,10398,5,14.777233,4.518158,5.987347


---
## 문제 9: 설비 고장 예측을 위한 특성 생성

**시나리오**: 설비 정비 이력과 생산/센서 데이터를 결합하여 고장 예측 특성을 생성하세요.

**요구사항**:
1. 설비별 정비 이력 집계:
   - 총 정비 건수
   - 총 정지 시간
   - 고장 정비 건수 (maintenance_type='BREAKDOWN')
2. 설비별 생산 집계:
   - 평균 사이클 타임
   - 평균 불량률
3. 설비별 센서 집계 (최근 30일):
   - 평균 온도
   - 평균 진동
   - 온도 표준편차
4. equipment_df에 위 세 집계를 모두 결합
5. 위험도 점수 계산:
   - 위험도 = (고장건수 × 10) + (평균불량률 × 5) + (온도표준편차 × 2)
6. 위험도 상위 5개 설비 출력

**힌트**: 각각 집계 후 순차적 merge, 복합 계산

In [105]:
# 여기에 코드 작성
maintenance_df.head()

,equipment_id,maintenance_type,start_time,end_time,maintenance_description,parts_replaced,cost,technician_id,downtime_hours,notes,created_at
0,INJ-002,PREVENTIVE,2024-01-03 10:06:38,2024-01-03 11:38:28,월간 정기점검,NaN,1597990.91,OP015,1.53,다음 정비 시 추가 점검 요망,2026-01-30 02:23:22.393125
1,INJ-002,PREVENTIVE,2024-01-05 15:47:52,2024-01-05 17:47:18,정기 윤활유 교환,"금형, 압력센서",2488706.64,OP011,1.99,NaN,2026-01-30 02:23:22.396196
2,PRESS-002,PREVENTIVE,2024-01-06 21:08:07,2024-01-06 22:49:13,정기 윤활유 교환,금형,1330618.46,OP017,1.69,NaN,2026-01-30 02:23:22.398028
3,ASM-001,PREVENTIVE,2024-01-07 08:00:00,2024-01-07 09:11:21,월간 정기점검,NaN,755786.19,OP017,1.19,NaN,2026-01-30 02:23:22.399504
4,INJ-001,BREAKDOWN,2024-01-08 12:30:24,2024-01-08 14:23:11,설비 고장으로 긴급 수리,"금형, 온도센서",5046550.70,OP013,1.88,NaN,2026-01-30 02:23:22.401063


In [108]:
def breakdown_count(x) :
    return (x=='BREAKDOWN').sum()

In [110]:
main_features=maintenance_df.groupby('equipment_id').agg({'equipment_id':'count',
                                            'downtime_hours':'sum',
                                            'maintenance_type':breakdown_count})

In [111]:
main_features.columns=['정비건수','총정지시간', '고장건수']

In [113]:
main_features

,정비건수,총정지시간,고장건수
equipment_id,,,
ASM-001,19,30.79,3
INJ-001,17,27.85,2
INJ-002,23,40.36,5
PRESS-001,18,29.69,2
PRESS-002,21,37.71,4


In [116]:
production_df.columns

Index(['production_id', 'equipment_id', 'product_code', 'production_date',
       'start_time', 'end_time', 'target_quantity', 'actual_quantity',
       'good_quantity', 'defect_quantity', 'cycle_time', 'work_order_no',
       'lot_no', 'operator_id', 'shift', 'created_at', 'updated_at', '양품률',
       '실제작업시간(분)', '성능률', 'OEE', 'year_month'],
      dtype='object')

In [118]:
prod_features=production_df.groupby('equipment_id').agg({'cycle_time':'mean',
                                           'defect_quantity':'sum',
                                           'actual_quantity':'sum'})

In [119]:
prod_features['평균불량률']=prod_features['defect_quantity']/prod_features['actual_quantity']*100

In [122]:
prod_features.columns =['평균사이클타임','평균불량갯수','총생산갯수','평균불량률']

In [123]:
prod_features

,평균사이클타임,평균불량갯수,총생산갯수,평균불량률
equipment_id,,,,
ASM-001,94.427137,2726,22485,12.123638
INJ-001,71.714695,3017,28163,10.712637
INJ-002,77.825558,4504,51958,8.668540
PRESS-001,73.568333,5123,52069,9.838868
PRESS-002,72.555900,5506,51929,10.602939


In [127]:
#최근 30일 데이터
start_time=sensor_df['measurement_time'].max() - pd.Timedelta(days=30) # 날자데이터 뺄셈

In [130]:
sensor_recent=sensor_df[sensor_df['measurement_time']>=start_time]

In [134]:
sensor_features=sensor_recent.groupby('equipment_id').agg({'temperature':['mean','std'],
                                           'vibration' : 'mean'})

In [135]:
sensor_features.columns = ['평균온도','온도표준편차','평균진동']

In [136]:
main_features.head(2)

,정비건수,총정지시간,고장건수
equipment_id,,,
ASM-001,19,30.79,3
INJ-001,17,27.85,2


In [137]:
prod_features.head(2)

,평균사이클타임,평균불량갯수,총생산갯수,평균불량률
equipment_id,,,,
ASM-001,94.427137,2726,22485,12.123638
INJ-001,71.714695,3017,28163,10.712637


In [139]:
main_comb=pd.merge(main_features,prod_features,on='equipment_id',how='left')

In [141]:
main_comb=pd.merge(main_comb,sensor_features,on='equipment_id',how='left')

In [142]:
(main_comb).round(2)

,정비건수,총정지시간,고장건수,평균사이클타임,평균불량갯수,총생산갯수,평균불량률,평균온도,온도표준편차,평균진동
equipment_id,,,,,,,,,,
ASM-001,19,30.79,3,94.43,2726,22485,12.12,25.02,2.24,1.20
INJ-001,17,27.85,2,71.71,3017,28163,10.71,192.54,2.55,2.50
INJ-002,23,40.36,5,77.83,4504,51958,8.67,174.93,2.23,2.29
PRESS-001,18,29.69,2,73.57,5123,52069,9.84,85.05,2.20,3.49
PRESS-002,21,37.71,4,72.56,5506,51929,10.60,88.09,2.10,4.08


In [144]:
# 위험도 계산
(main_comb['고장건수']*10 + main_comb['평균불량률']*5 + main_comb['온도표준편차']*2).sort_values(ascending=False)

equipment_id
INJ-002      97.811325
PRESS-002    97.213065
ASM-001      95.099099
INJ-001      78.670628
PRESS-001    73.592079
dtype: float64

---
## 문제 10: 종합 제조 대시보드 생성

**시나리오**: 경영진을 위한 종합 제조 대시보드 데이터를 생성하세요.

**요구사항**:

### Part A: 전체 현황 (최근 30일)
1. 최근 30일 데이터 필터링
2. 다음 KPI 계산:
   - 총 생산량
   - 평균 일 생산량
   - 전체 불량률
   - 가동 설비 수
   - 평균 OEE

### Part B: 설비 타입별 분석
1. equipment_df와 production_df 결합
2. 설비 타입별로 집계:
   - 설비 수
   - 총 생산량
   - 평균 불량률
   - 평균 OEE

### Part C: Top/Bottom 설비
1. OEE 상위 3개 설비
2. 불량률 하위 3개 설비 (낮을수록 좋음)
3. 생산량 상위 3개 설비

### Part D: 일별 추이 (최근 30일)
1. 일별 생산량 및 7일 이동평균
2. 일별 불량률 및 7일 이동평균

모든 결과를 출력하세요.

**힌트**: 날짜 필터링, 복합 집계, merge, rolling, 종합적 데이터 처리

In [56]:
# 여기에 코드 작성

# Part A: 전체 현황


# Part B: 설비 타입별 분석


# Part C: Top/Bottom 설비


# Part D: 일별 추이


---
## 수고하셨습니다!

### 학습 완료 체크리스트
- [ ] 시계열 데이터 집계 (resample)
- [ ] 이동평균 계산 (rolling)
- [ ] 변화율 및 차분 분석 (shift, diff, pct_change)
- [ ] 이상치 탐지 (3-Sigma, IQR)
- [ ] OEE 계산 및 설비 효율 분석
- [ ] SPC 관리도 데이터 생성
- [ ] 다중 테이블 결합 및 종합 분석
- [ ] 실무 대시보드 데이터 구축

